In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
report_path = "results/eval_3APR_4.json"

eval_report = pd.read_json(f'../{report_path}')

In [ ]:
total_cases = len(eval_report)

hallucination_count = (eval_report['fch']  & eval_report['ich']).sum()
hallucination_rate = hallucination_count / total_cases * 100

fn_count = (eval_report['fn'] == True).sum()
fnr = fn_count / total_cases * 100

avg_score = eval_report.loc[~eval_report['fn'], 'grade'].mean()
score_std = eval_report.loc[~eval_report['fn'], 'grade'].std()

ood = eval_report[eval_report['category'] == 'out_of_domain']
ood_violations_count = ood['ich'].sum()
ood_violations_rate = ood_violations_count/ len(ood) * 100

print(f"--- Global Metrics ---")
print(f"Number of cases evaluated: {total_cases}")
print(f"Hallucinations: {hallucination_count} ({hallucination_rate:.1f}%)")
print(f"False Negatives: {fn_count} ({fnr:.1f}%)")
print(f"Out-of-domain violations: {ood_violations_count} ({ood_violations_rate:.1f}%)")
print(f"Average Safety Score for positive responses (in and out of domain): {avg_score:.2f}/5.0 (+/- {score_std:.2f})")


In [ ]:
# split the category into complexity and language
eval_report[['complexity', 'formality']] = eval_report['category'].apply(lambda x: pd.Series(['', ''] if x=='out_of_domain' else x.split('_')))

hall_type = pd.CategoricalDtype(["No hallucination", "ICH only", "FCH only", "Both FCH and ICH"], ordered=True)

no_h = ~eval_report['ich'] & ~eval_report['fch']
eval_report.loc[no_h, 'Hallucination'] = hall_type.categories[0]
ich_only = eval_report['ich'] & ~eval_report['fch']
eval_report.loc[ich_only, 'Hallucination'] = hall_type.categories[1]
fch_only = ~eval_report['ich'] & eval_report['fch']
eval_report.loc[fch_only, 'Hallucination'] = hall_type.categories[2]
both_h = eval_report['ich'] & eval_report['fch']
eval_report.loc[both_h, 'Hallucination'] = hall_type.categories[3]

eval_report['Hallucination'] = eval_report['Hallucination'].astype(hall_type)

in_domain = eval_report[eval_report['category'] != 'out_of_domain']
positive_responses_in_domain = in_domain[~in_domain['fn']]

g = sns.catplot(positive_responses_in_domain, kind="bar", x="Hallucination", y="grade", col='complexity', hue='formality')
g.set_xlabels("")
plt.show()

## Review cases with hallucinations and safety score > 3

In [ ]:
import textwrap

wrapper = textwrap.TextWrapper(width=120)

hall_high_score = positive_responses_in_domain[
  positive_responses_in_domain['grade'] > 3 & (positive_responses_in_domain['fch'] | positive_responses_in_domain['ich'])
  ]

print(f"Cases with hallucination and high safety score: {len(hall_high_score)}")
print(f'Cases to review: {hall_high_score['id'].values.tolist()}')  
